# Amazon Business Entity Resolution: High-Recall Blocking Optimization
### Objective: Maximize Blocking Recall (Target >=99%) on 50k Pilot Benchmark
- Audit 13.31% missed matches from baseline across 18 specific failure modes.
- Evaluate individual blocker contributions and unique value.
- Progressively evaluate Versions 1 through 8 (Char N-Grams, Rare Token Pairs, Address Signatures, Phonetic Retrieval, Uncapped Indexes).
- Validate frozen blocker on independent 50k Pilot B.
- Save checkpoints and analytical reports.

In [ ]:
import os
import sys
import shutil
import psutil
from pathlib import Path

print('=== ENVIRONMENT & RESOURCE CHECK ===')
ram = psutil.virtual_memory()
print(f'Total RAM: {ram.total / (1024**3):.2f} GB | Available: {ram.available / (1024**3):.2f} GB')
print(f'CPU Count: {os.cpu_count()}')

# Setup working directory structure
BASE_WORKING = Path('/kaggle/working/business_er') if Path('/kaggle/working').exists() else Path('./business_er')
for sub in ['src', 'checkpoints/blocking', 'cache', 'reports', 'output', 'logs']:
    (BASE_WORKING / sub).mkdir(parents=True, exist_ok=True)

SRC_DIR = BASE_WORKING / 'src'
sys.path.insert(0, str(SRC_DIR))
print(f'Workspace initialized at: {BASE_WORKING}')


In [ ]:
# Writing src/normalization.py
norm_code = "\"\"\"\nStage 1: Advanced Normalization Pipeline (Enhanced for High-Recall Blocking)\nMaintains both raw and multiple specialized representations of names and addresses.\nSupports country-aware variations for India, US, and France (open set).\nExtracts structured components (postal code, house number, numeric tokens, legal suffixes, phonetic tokens, rare tokens).\n\"\"\"\n\nimport re\nimport unicodedata\nfrom typing import Dict, List, Set, Tuple, Optional\n\n# Extended legal suffixes across US, India, and France\nLEGAL_SUFFIXES_LIST = [\n    # General / US / UK\n    \"LLC\", \"INC\", \"INCORPORATED\", \"CORP\", \"CORPORATION\", \"LTD\", \"LIMITED\",\n    \"CO\", \"COMPANY\", \"LLP\", \"PLC\", \"HOLDINGS\", \"GROUP\", \"ENTERPRISES\", \"INDUSTRIES\",\n    # India\n    \"PVT\", \"PRIVATE\", \"PVT LTD\", \"PRIVATE LIMITED\", \"LTD CO\",\n    # France / Continental Europe\n    \"SARL\", \"SAS\", \"SA\", \"EURL\", \"SNC\", \"SCI\", \"SCOP\", \"GIE\", \"GMBH\"\n]\n\n# Regex for stripping legal suffixes safely\n_SUFFIX_PATTERN = r'\\b(' + '|'.join(sorted(LEGAL_SUFFIXES_LIST, key=len, reverse=True)) + r')\\b'\nLEGAL_RE = re.compile(_SUFFIX_PATTERN, re.IGNORECASE)\n\n# Standard abbreviation mappings (US, India, UK, France, general)\nCOMMON_ABBREVIATIONS = {\n    # Corporate\n    \"CORP\": \"CORPORATION\", \"INC\": \"INCORPORATED\", \"PVT\": \"PRIVATE\", \"LTD\": \"LIMITED\",\n    \"CO\": \"COMPANY\", \"BROS\": \"BROTHERS\", \"ASSN\": \"ASSOCIATION\", \"MFG\": \"MANUFACTURING\",\n    \"DIST\": \"DISTRIBUTION\", \"INTL\": \"INTERNATIONAL\", \"NATL\": \"NATIONAL\", \"TECH\": \"TECHNOLOGY\",\n    \"INFO\": \"INFORMATION\", \"SYS\": \"SYSTEMS\", \"COMM\": \"COMMUNICATIONS\", \"MED\": \"MEDICAL\",\n    \"PHARM\": \"PHARMACEUTICAL\", \"DEV\": \"DEVELOPMENT\", \"MKTG\": \"MARKETING\", \"CONS\": \"CONSULTING\",\n    \"ENG\": \"ENGINEERING\", \"SVCS\": \"SERVICES\", \"TRDG\": \"TRADING\", \"ENT\": \"ENTERPRISES\",\n    \"IND\": \"INDUSTRIES\", \"DEPT\": \"DEPARTMENT\",\n    # Address / US & UK\n    \"ST\": \"STREET\", \"RD\": \"ROAD\", \"AVE\": \"AVENUE\", \"BLVD\": \"BOULEVARD\",\n    \"DR\": \"DRIVE\", \"LN\": \"LANE\", \"HWY\": \"HIGHWAY\", \"SQ\": \"SQUARE\", \"PL\": \"PLACE\",\n    \"CT\": \"COURT\", \"STE\": \"SUITE\", \"APT\": \"APARTMENT\", \"FL\": \"FLOOR\", \"CTR\": \"CENTER\",\n    \"PKWY\": \"PARKWAY\", \"CIR\": \"CIRCLE\", \"WAY\": \"WAY\", \"TER\": \"TERRACE\", \"BLDG\": \"BUILDING\",\n    # Address / France\n    \"RUE\": \"RUE\", \"BD\": \"BOULEVARD\", \"AV\": \"AVENUE\", \"IMP\": \"IMPASSE\", \"ALL\": \"ALLEE\",\n    \"CHE\": \"CHEMIN\", \"RTE\": \"ROUTE\", \"ZA\": \"ZONE\", \"ZI\": \"ZONE\", \"ZAC\": \"ZONE\",\n    \"SOC\": \"SOCIETE\", \"ETS\": \"ETABLISSEMENTS\"\n}\n\nGENERIC_STOPWORDS = {\n    \"the\", \"and\", \"of\", \"in\", \"for\", \"to\", \"at\", \"by\", \"on\", \"a\", \"an\",\n    \"co\", \"company\", \"services\", \"service\", \"enterprises\", \"enterprise\",\n    \"group\", \"holdings\", \"holding\", \"solutions\", \"solution\", \"systems\", \"system\",\n    \"associates\", \"consultants\", \"consulting\", \"management\", \"international\", \"national\",\n    \"india\", \"us\", \"usa\", \"france\", \"corporation\", \"limited\", \"private\", \"llc\", \"inc\", \"pvt\", \"ltd\"\n}\n\n# Indian transliteration phonetic mappings\nINDIAN_PHONETIC_PAIRS = [\n    (r'ee', 'i'),\n    (r'oo', 'u'),\n    (r'w', 'v'),\n    (r'sh', 's'),\n    (r'ch', 'c'),\n    (r'dh', 'd'),\n    (r'th', 't'),\n    (r'bh', 'b'),\n    (r'ph', 'f'),\n    (r'kh', 'k'),\n    (r'gh', 'g'),\n    (r'jh', 'j')\n]\n\n\ndef strip_accents(text: str) -> str:\n    \"\"\"Remove diacritics/accents safely using NFKD decomposition.\"\"\"\n    if not text:\n        return \"\"\n    nfkd = unicodedata.normalize('NFKD', text)\n    return \"\".join(c for c in nfkd if not unicodedata.combining(c))\n\n\ndef extract_char_ngrams(text: str, n: int) -> Set[str]:\n    \"\"\"Extract character n-grams from a normalized string.\"\"\"\n    clean = re.sub(r'\\s+', ' ', text).strip()\n    if len(clean) < n:\n        return {clean} if clean else set()\n    return {clean[i:i+n] for i in range(len(clean) - n + 1)}\n\n\ndef get_consonant_skeleton(word: str, country: str = \"\") -> str:\n    \"\"\"\n    Compute consonant skeleton with phonetic mapping and duplicate letter collapsing.\n    \"\"\"\n    if not word:\n        return \"\"\n    w = word.lower().strip()\n    if country == \"India\" or True:\n        for pat, rep in INDIAN_PHONETIC_PAIRS:\n            w = re.sub(pat, rep, w)\n    first = w[0]\n    rest = re.sub(r'[aeiouy\\s]', '', w[1:])\n    combined = first + rest\n    # collapse consecutive duplicates\n    collapsed = re.sub(r'(.)\\1+', r'\\1', combined)\n    return collapsed\n\n\ndef soundex(word: str) -> str:\n    \"\"\"Standard American Soundex encoding for phonetic blocking.\"\"\"\n    if not word or not word.isalpha():\n        return \"\"\n    w = word.upper()\n    mapping = {\n        'B': '1', 'F': '1', 'P': '1', 'V': '1',\n        'C': '2', 'G': '2', 'J': '2', 'K': '2', 'Q': '2', 'S': '2', 'X': '2', 'Z': '2',\n        'D': '3', 'T': '3',\n        'L': '4',\n        'M': '5', 'N': '5',\n        'R': '6'\n    }\n    first = w[0]\n    encoded = []\n    prev_code = mapping.get(first, '')\n    for char in w[1:]:\n        code = mapping.get(char, '')\n        if code and code != prev_code:\n            encoded.append(code)\n            prev_code = code\n        elif not code and char not in 'HW':\n            prev_code = ''\n    res = (first + ''.join(encoded) + '000')[:4]\n    return res\n\n\ndef normalize_business_name(raw_name: Optional[str], country: str = \"US\") -> Dict:\n    \"\"\"\n    Generate multiple representations and extracted features for a business name.\n    Preserves raw name while providing normalized views for blocking and matching.\n    \"\"\"\n    raw = str(raw_name or \"\").strip()\n    if not raw:\n        return {\n            \"name_raw\": \"\",\n            \"name_nfkc\": \"\",\n            \"name_lower\": \"\",\n            \"name_ascii_when_safe\": \"\",\n            \"name_alnum\": \"\",\n            \"name_tokens\": [],\n            \"name_tokens_sorted\": \"\",\n            \"name_token_set\": set(),\n            \"name_no_legal_suffix\": \"\",\n            \"name_no_legal_suffix_sorted\": \"\",\n            \"name_initials\": \"\",\n            \"name_acronym\": \"\",\n            \"legal_suffixes\": [],\n            \"name_char_3gram\": set(),\n            \"name_char_4gram\": set(),\n            \"name_char_5gram\": set(),\n            \"distinctive_tokens\": [],\n            \"phonetic_tokens\": [],\n            \"soundex_tokens\": [],\n            \"name_abbrev_expanded\": \"\",\n        }\n\n    # Unicode NFKC\n    nfkc = unicodedata.normalize('NFKC', raw)\n    # Lowercase\n    lower = nfkc.lower()\n    # Ascii-clean\n    ascii_safe = strip_accents(lower)\n\n    # Extract detected legal suffixes\n    legal_matches = [m.upper() for m in LEGAL_RE.findall(ascii_safe)]\n\n    # Alphanumeric clean\n    alnum = re.sub(r'[^a-z0-9\\s]', ' ', ascii_safe)\n    alnum = re.sub(r'\\s+', ' ', alnum).strip()\n\n    # Tokens\n    tokens = [t for t in alnum.split() if t]\n    tokens_sorted = \" \".join(sorted(tokens))\n    token_set = set(tokens)\n\n    # Tokens without legal suffix\n    no_suffix_tokens = [t for t in tokens if t.upper() not in LEGAL_SUFFIXES_LIST]\n    no_suffix = \" \".join(no_suffix_tokens)\n    no_suffix_sorted = \" \".join(sorted(no_suffix_tokens))\n\n    # Distinctive (non-generic) tokens\n    distinctive_tokens = [t for t in no_suffix_tokens if t not in GENERIC_STOPWORDS and len(t) >= 3]\n\n    # Abbreviation expansion\n    words_expanded = [COMMON_ABBREVIATIONS.get(t.upper(), t.upper()).lower() for t in tokens]\n    name_abbrev_expanded = \" \".join(words_expanded)\n\n    # Phonetic tokens & soundex\n    phonetic_tokens = [get_consonant_skeleton(t, country=country) for t in distinctive_tokens if len(t) >= 3]\n    soundex_tokens = [soundex(t) for t in distinctive_tokens if len(t) >= 3 and soundex(t)]\n\n    # Initials / Acronym\n    initials = \"\".join([t[0] for t in tokens]) if tokens else \"\"\n    acronym = initials.upper()\n\n    # Character n-grams\n    c3 = extract_char_ngrams(alnum, 3)\n    c4 = extract_char_ngrams(alnum, 4)\n    c5 = extract_char_ngrams(alnum, 5)\n\n    return {\n        \"name_raw\": raw,\n        \"name_nfkc\": nfkc,\n        \"name_lower\": lower,\n        \"name_ascii_when_safe\": ascii_safe,\n        \"name_alnum\": alnum,\n        \"name_tokens\": tokens,\n        \"name_tokens_sorted\": tokens_sorted,\n        \"name_token_set\": token_set,\n        \"name_no_legal_suffix\": no_suffix,\n        \"name_no_legal_suffix_sorted\": no_suffix_sorted,\n        \"name_initials\": initials,\n        \"name_acronym\": acronym,\n        \"legal_suffixes\": legal_matches,\n        \"name_char_3gram\": c3,\n        \"name_char_4gram\": c4,\n        \"name_char_5gram\": c5,\n        \"distinctive_tokens\": distinctive_tokens,\n        \"phonetic_tokens\": phonetic_tokens,\n        \"soundex_tokens\": soundex_tokens,\n        \"name_abbrev_expanded\": name_abbrev_expanded,\n    }\n\n\ndef extract_postal_code(text: str, country: str = \"\") -> Optional[str]:\n    \"\"\"\n    Extract postal code in a country-aware manner:\n    - India: 6-digit PIN code (e.g. 110001, 400001)\n    - US: 5-digit ZIP code (e.g. 90210, 02138)\n    - France: 5-digit code postal (e.g. 75001, 69002)\n    \"\"\"\n    if country == \"India\":\n        m = re.findall(r'\\b[1-9][0-9]{5}\\b', text)\n        if m:\n            return m[-1]\n    elif country in [\"US\", \"France\"]:\n        m = re.findall(r'\\b[0-9]{5}\\b', text)\n        if m:\n            return m[-1]\n    else:\n        m = re.findall(r'\\b[0-9]{5,6}\\b', text)\n        if m:\n            return m[-1]\n    return None\n\n\ndef extract_house_number(tokens: List[str]) -> Optional[str]:\n    \"\"\"Extract leading or prominent street/house number.\"\"\"\n    if not tokens:\n        return None\n    if re.match(r'^\\d+[a-z]?$', tokens[0]):\n        return tokens[0]\n    for t in tokens[:4]:\n        if re.match(r'^\\d+$', t):\n            return t\n    return None\n\n\ndef normalize_business_address(raw_address: Optional[str], country: str = \"US\") -> Dict:\n    \"\"\"\n    Generate multiple representations and structured components for an address.\n    \"\"\"\n    raw = str(raw_address or \"\").strip()\n    if not raw:\n        return {\n            \"address_raw\": \"\",\n            \"address_nfkc\": \"\",\n            \"address_lower\": \"\",\n            \"address_alnum\": \"\",\n            \"address_tokens\": [],\n            \"address_tokens_sorted\": \"\",\n            \"postal_code\": None,\n            \"house_number\": None,\n            \"numeric_tokens\": set(),\n            \"address_char_3gram\": set(),\n            \"address_char_4gram\": set(),\n            \"address_char_5gram\": set(),\n            \"address_distinctive_tokens\": [],\n            \"street_name_tokens\": []\n        }\n\n    nfkc = unicodedata.normalize('NFKC', raw)\n    lower = nfkc.lower()\n    ascii_safe = strip_accents(lower)\n\n    # Expand standard abbreviations\n    words = re.findall(r'\\b\\w+\\b', ascii_safe.upper())\n    expanded_words = [COMMON_ABBREVIATIONS.get(w, w) for w in words]\n    expanded_text = \" \".join(expanded_words).lower()\n\n    # Alphanumeric clean\n    alnum = re.sub(r'[^a-z0-9\\s]', ' ', expanded_text)\n    alnum = re.sub(r'\\s+', ' ', alnum).strip()\n\n    tokens = [t for t in alnum.split() if t]\n    tokens_sorted = \" \".join(sorted(tokens))\n\n    # Structured components\n    postal = extract_postal_code(raw, country=country)\n    digits = set(re.findall(r'\\b\\d+\\b', raw))\n    house_num = extract_house_number(tokens)\n\n    # Address distinctive tokens (non-numeric, length >= 4)\n    addr_distinctive = [t for t in tokens if not t.isdigit() and len(t) >= 4 and t not in GENERIC_STOPWORDS]\n\n    # Character ngrams\n    c3 = extract_char_ngrams(alnum, 3)\n    c4 = extract_char_ngrams(alnum, 4)\n    c5 = extract_char_ngrams(alnum, 5)\n\n    return {\n        \"address_raw\": raw,\n        \"address_nfkc\": nfkc,\n        \"address_lower\": lower,\n        \"address_alnum\": alnum,\n        \"address_tokens\": tokens,\n        \"address_tokens_sorted\": tokens_sorted,\n        \"postal_code\": postal,\n        \"house_number\": house_num,\n        \"numeric_tokens\": digits,\n        \"address_char_3gram\": c3,\n        \"address_char_4gram\": c4,\n        \"address_char_5gram\": c5,\n        \"address_distinctive_tokens\": addr_distinctive,\n        \"street_name_tokens\": addr_distinctive[:3]\n    }\n"
with open(SRC_DIR / 'normalization.py', 'w', encoding='utf-8') as f:
    f.write(norm_code)
print('Wrote normalization.py successfully.')


In [ ]:
# Writing src/blocking.py
block_code = "\"\"\"\nStage 2: High-Recall Multi-Strategy Blocking Engine (V2 - Target >=99% Recall)\nImplements 20+ specialized high-recall, precision-conscious blocking strategies.\nEliminates arbitrary posting-list cutoffs that lose true matches.\nEmploys country partitioning, rare-token indexing, character n-gram inverted indexing,\nphonetic representations, address signatures, and adaptive key frequency filtering.\n\"\"\"\n\nimport sys\nimport time\nimport re\nfrom collections import defaultdict, Counter\nfrom typing import Dict, List, Set, Tuple, Optional\nimport polars as pl\nimport numpy as np\n\nfrom normalization import (\n    normalize_business_name,\n    normalize_business_address,\n    extract_postal_code,\n    extract_house_number,\n    get_consonant_skeleton,\n    soundex,\n    GENERIC_STOPWORDS\n)\n\n# Key frequency threshold: keys appearing more than this across the source are skipped\n# as uninformative stopwords to avoid memory explosion, while informative keys retain 100% of records.\nMAX_KEY_FREQUENCY = 15000\n\n\nclass HighRecallBlocker:\n    \"\"\"\n    High-Recall Partitioned Inverted Index Blocker.\n    Supports individual blocker ablation, cumulative union, and unique contribution auditing.\n    \"\"\"\n\n    def __init__(self, max_key_frequency: int = MAX_KEY_FREQUENCY):\n        self.max_key_frequency = max_key_frequency\n        # index: strategy -> country -> key -> list of candidate_ids\n        self.indices = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))\n        self.key_counts = defaultdict(lambda: defaultdict(Counter))\n        self.indexed_candidate_count = 0\n\n    def generate_keys_for_record(\n        self,\n        name_norm: dict,\n        addr_norm: dict,\n        country: str,\n        active_versions: Optional[Set[int]] = None\n    ) -> Dict[str, List[str]]:\n        \"\"\"\n        Generate blocking keys for all implemented strategies.\n        Optionally filter by version level (1 to 8).\n        \"\"\"\n        keys = defaultdict(list)\n        cntry = country or \"UNKNOWN\"\n\n        name_alnum = name_norm[\"name_alnum\"]\n        no_suf = name_norm[\"name_no_legal_suffix\"]\n        tokens = name_norm[\"name_tokens\"]\n        sorted_tokens = name_norm[\"name_tokens_sorted\"]\n        initials = name_norm[\"name_initials\"]\n        c3_grams = name_norm[\"name_char_3gram\"]\n        c4_grams = name_norm[\"name_char_4gram\"]\n        distinctive_toks = name_norm[\"distinctive_tokens\"]\n        phonetic_toks = name_norm[\"phonetic_tokens\"]\n        soundex_toks = name_norm[\"soundex_tokens\"]\n        name_abbrev = name_norm[\"name_abbrev_expanded\"]\n\n        addr_alnum = addr_norm[\"address_alnum\"]\n        addr_tokens = addr_norm[\"address_tokens\"]\n        postal = addr_norm[\"postal_code\"]\n        house_num = addr_norm[\"house_number\"]\n        numeric_tokens = list(addr_norm[\"numeric_tokens\"])\n        addr_distinctive = addr_norm[\"address_distinctive_tokens\"]\n\n        # -------------------------------------------------------------\n        # VERSION 1: BASELINE STRATEGIES\n        # -------------------------------------------------------------\n        if active_versions is None or 1 in active_versions:\n            # 1. Exact normalized name\n            if name_alnum:\n                keys[\"B01_exact_name\"].append(name_alnum)\n\n            # 2. Name without legal suffix\n            if no_suf and no_suf != name_alnum:\n                keys[\"B02_no_legal_suffix\"].append(no_suf)\n\n            # 3. Sorted name tokens\n            if sorted_tokens and sorted_tokens != name_alnum:\n                keys[\"B03_sorted_tokens\"].append(sorted_tokens)\n\n            # 4. Initials + first token prefix\n            if len(tokens) >= 2 and len(initials) >= 2:\n                keys[\"B04_initials_pfx\"].append(f\"{initials[:4]}_{tokens[0][:4]}\")\n\n            # 5. First two tokens\n            if len(tokens) >= 2:\n                keys[\"B05_first_two_tokens\"].append(f\"{tokens[0]}_{tokens[1]}\")\n\n            # 6. First 4 chars + length bucket\n            if len(name_alnum) >= 4:\n                keys[\"B06_prefix4_len\"].append(f\"{name_alnum[:4]}_{len(name_alnum)//4}\")\n\n            # 7. House number + first name token\n            if house_num and tokens:\n                keys[\"B07_house_tok0\"].append(f\"{house_num}_{tokens[0]}\")\n\n            # 8. Postal code + 3-char name prefix\n            if postal and len(name_alnum) >= 3:\n                keys[\"B08_postal_name_pfx\"].append(f\"{postal}_{name_alnum[:3]}\")\n\n            # 9. Postal code + House number\n            if postal and house_num:\n                keys[\"B09_postal_house\"].append(f\"{postal}_{house_num}\")\n\n            # 10. First name token + numeric token\n            if tokens and numeric_tokens:\n                keys[\"B10_token_numeric\"].append(f\"{tokens[0]}_{numeric_tokens[0]}\")\n\n            # 11. Address leading tokens\n            if len(addr_tokens) >= 2:\n                keys[\"B11_addr_lead_tokens\"].append(f\"{addr_tokens[0]}_{addr_tokens[1]}\")\n\n        # -------------------------------------------------------------\n        # VERSION 2: FIX NORMALIZATION & COMPOUND WORDS & ABBREVIATIONS\n        # -------------------------------------------------------------\n        if active_versions is None or 2 in active_versions:\n            # 12. Abbreviation-expanded name\n            if name_abbrev and name_abbrev != name_alnum:\n                keys[\"B12_name_abbrev_expanded\"].append(name_abbrev)\n\n            # 13. Compound / concatenated name without spaces (e.g., \"wal mart\" vs \"walmart\")\n            name_no_space = \"\".join(tokens)\n            if len(name_no_space) >= 5 and name_no_space != name_alnum:\n                keys[\"B13_name_no_space\"].append(name_no_space)\n\n            # 14. First distinctive non-stopword token (handles \"The Home Depot\" -> \"home\")\n            if distinctive_toks:\n                keys[\"B14_first_distinctive_token\"].append(distinctive_toks[0])\n\n        # -------------------------------------------------------------\n        # VERSION 3: CHARACTER N-GRAM SHINGLES (RECOVERS TYPOS & EDITS)\n        # -------------------------------------------------------------\n        if active_versions is None or 3 in active_versions:\n            # 15. Prefix 4-gram + Suffix 4-gram\n            if len(name_alnum) >= 8:\n                keys[\"B15_prefix_suffix_4gram\"].append(f\"{name_alnum[:4]}_{name_alnum[-4:]}\")\n\n            # 16. Skip-gram: first 2 chars + middle 2 chars\n            if len(name_alnum) >= 6:\n                mid = len(name_alnum) // 2\n                keys[\"B16_skip_gram\"].append(f\"{name_alnum[:2]}_{name_alnum[mid:mid+2]}_{name_alnum[-2:]}\")\n\n            # 17. Two rarest 3-grams from distinctive tokens\n            if len(distinctive_toks) >= 1:\n                t0 = distinctive_toks[0]\n                if len(t0) >= 5:\n                    keys[\"B17_distinctive_tok_ngrams\"].append(f\"{t0[:3]}_{t0[-3:]}\")\n\n        # -------------------------------------------------------------\n        # VERSION 4: RARE-TOKEN BLOCKS & TOKEN PAIRS (RECOVERS WORD REORDERS)\n        # -------------------------------------------------------------\n        if active_versions is None or 4 in active_versions:\n            # 18. Any pair of distinctive tokens\n            if len(distinctive_toks) >= 2:\n                for i in range(min(3, len(distinctive_toks))):\n                    for j in range(i + 1, min(4, len(distinctive_toks))):\n                        pair = sorted([distinctive_toks[i], distinctive_toks[j]])\n                        keys[\"B18_distinctive_token_pair\"].append(f\"{pair[0]}_{pair[1]}\")\n\n            # 19. First + Last distinctive token\n            if len(distinctive_toks) >= 2:\n                keys[\"B19_first_last_distinctive\"].append(f\"{distinctive_toks[0]}_{distinctive_toks[-1]}\")\n\n            # 20. Second distinctive token + house number\n            if len(distinctive_toks) >= 2 and house_num:\n                keys[\"B20_tok1_house\"].append(f\"{distinctive_toks[1]}_{house_num}\")\n\n        # -------------------------------------------------------------\n        # VERSION 5: ADDRESS SIGNATURES & LOCALITY\n        # -------------------------------------------------------------\n        if active_versions is None or 5 in active_versions:\n            # 21. Postal code + first distinctive name token\n            if postal and distinctive_toks:\n                keys[\"B21_postal_distinctive_tok\"].append(f\"{postal}_{distinctive_toks[0]}\")\n\n            # 22. Street distinctive token + first distinctive name token\n            if addr_distinctive and distinctive_toks:\n                keys[\"B22_street_name_joint\"].append(f\"{addr_distinctive[0]}_{distinctive_toks[0]}\")\n\n            # 23. Numeric token signature + first distinctive token\n            if len(numeric_tokens) >= 2 and distinctive_toks:\n                sorted_nums = \"_\".join(sorted(numeric_tokens)[:2])\n                keys[\"B23_numeric_sig_name\"].append(f\"{sorted_nums}_{distinctive_toks[0]}\")\n\n        # -------------------------------------------------------------\n        # VERSION 6: PHONETIC RETRIEVAL (INDIAN TRANSLITERATIONS & PHONETICS)\n        # -------------------------------------------------------------\n        if active_versions is None or 6 in active_versions:\n            # 24. Consonant skeleton of first distinctive token\n            if phonetic_toks:\n                keys[\"B24_phonetic_tok0\"].append(phonetic_toks[0][:6])\n\n            # 25. Phonetic pair (first 2 distinctive tokens)\n            if len(phonetic_toks) >= 2:\n                p_pair = sorted([phonetic_toks[0][:5], phonetic_toks[1][:5]])\n                keys[\"B25_phonetic_pair\"].append(f\"{p_pair[0]}_{p_pair[1]}\")\n\n            # 26. Soundex token 0 + postal code\n            if soundex_toks and postal:\n                keys[\"B26_soundex_postal\"].append(f\"{soundex_toks[0]}_{postal}\")\n\n            # 27. Consonant skeleton + house number\n            if phonetic_toks and house_num:\n                keys[\"B27_phonetic_house\"].append(f\"{phonetic_toks[0][:5]}_{house_num}\")\n\n        return keys\n\n    def index_candidate_source(self, df_candidates: pl.DataFrame, source_name: str):\n        \"\"\"\n        Index full candidate source into memory-efficient country-partitioned inverted indexes.\n        \"\"\"\n        print(f\"Indexing {len(df_candidates):,} records from {source_name}...\")\n        t0 = time.time()\n        count = 0\n\n        for row in df_candidates.select([\"entity_id\", \"business_name\", \"business_address\", \"country\"]).iter_rows():\n            ent_id = row[0]\n            raw_name = row[1]\n            raw_addr = row[2]\n            cntry = str(row[3] or \"UNKNOWN\")\n\n            name_norm = normalize_business_name(raw_name, country=cntry)\n            addr_norm = normalize_business_address(raw_addr, country=cntry)\n\n            keys_dict = self.generate_keys_for_record(name_norm, addr_norm, cntry)\n\n            for strat, key_list in keys_dict.items():\n                strat_index = self.indices[strat][cntry]\n                for k in key_list:\n                    plist = strat_index[k]\n                    # Never drop records arbitrarily; limit only huge generic uninformative keys\n                    if len(plist) < self.max_key_frequency:\n                        plist.append(ent_id)\n\n            count += 1\n            if count % 1000000 == 0:\n                print(f\"  Indexed {count:,} records in {time.time()-t0:.1f}s...\")\n\n        self.indexed_candidate_count += count\n        print(f\"Finished indexing {source_name} ({count:,} records) in {time.time() - t0:.2f}s.\")\n\n    def query_s1_entity(\n        self,\n        name_norm: dict,\n        addr_norm: dict,\n        country: str,\n        enabled_strategies: Optional[Set[str]] = None\n    ) -> Tuple[Set[str], Dict[str, Set[str]]]:\n        \"\"\"\n        Retrieve candidates for a single S1 entity.\n        Returns:\n            union_candidates: set of candidate IDs across all enabled strategies\n            strat_candidates: dict mapping strategy -> candidate IDs\n        \"\"\"\n        cntry = country or \"UNKNOWN\"\n        keys_dict = self.generate_keys_for_record(name_norm, addr_norm, cntry)\n\n        union_candidates = set()\n        strat_candidates = {}\n\n        for strat, key_list in keys_dict.items():\n            if enabled_strategies is not None and strat not in enabled_strategies:\n                continue\n\n            strat_index = self.indices[strat].get(cntry, {})\n            cands_for_strat = set()\n            for k in key_list:\n                posting = strat_index.get(k, [])\n                if posting:\n                    cands_for_strat.update(posting)\n\n            if cands_for_strat:\n                strat_candidates[strat] = cands_for_strat\n                union_candidates.update(cands_for_strat)\n\n        return union_candidates, strat_candidates\n"
with open(SRC_DIR / 'blocking.py', 'w', encoding='utf-8') as f:
    f.write(block_code)
print('Wrote blocking.py successfully.')


In [ ]:
# Writing src/evaluate_blocking_pipeline.py
pipeline_code = "\"\"\"\nEvaluation and Audit Script for High-Recall Blocking Optimization.\nExecutes on Kaggle compute (~31GB RAM, 4 vCPUs).\nImplements:\n- Step 1: Audit of missed matches from baseline (categorized into 18 failure modes) -> reports/blocking_miss_analysis.md\n- Step 2: Individual blocker audit & unique contribution\n- Step 7 & 8: Iterative recall improvement across Versions 1 to 8 -> reports/blocking_experiments.csv\n- Step 9 & 10: Validation on Pilot A (50k S1) and independent Pilot B (50k S1)\n- Step 12: Checkpointing to checkpoints/blocking/\n\"\"\"\n\nimport sys\nimport os\nimport gc\nimport time\nimport json\nimport csv\nimport re\nfrom pathlib import Path\nfrom collections import defaultdict, Counter\nimport numpy as np\nimport polars as pl\n\n# Ensure src is on python path\nsys.path.insert(0, str(Path(__file__).resolve().parent))\n\nfrom normalization import (\n    normalize_business_name,\n    normalize_business_address,\n    extract_postal_code,\n    extract_house_number\n)\nfrom blocking import HighRecallBlocker\n\n# Setup paths\nBASE_DIR = Path(__file__).resolve().parent.parent\nWORKING_DIR = Path(\"/kaggle/working/business_er\") if Path(\"/kaggle/working\").exists() else BASE_DIR\nREPORTS_DIR = WORKING_DIR / \"reports\"\nCHECKPOINTS_DIR = WORKING_DIR / \"checkpoints\" / \"blocking\"\nREPORTS_DIR.mkdir(parents=True, exist_ok=True)\nCHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)\n\n# Dataset path resolution\nDATA_SEARCH_ROOTS = [\n    Path(\"/kaggle/input\"),\n    BASE_DIR / \"student_resource\" / \"dataset\",\n    BASE_DIR / \"student_resource\",\n    BASE_DIR\n]\n\ndef find_file(filename: str) -> Path:\n    for root in DATA_SEARCH_ROOTS:\n        if root.exists():\n            matches = list(root.rglob(filename))\n            if matches:\n                return matches[0]\n    raise FileNotFoundError(f\"Required dataset file not found: {filename}\")\n\n\ndef categorize_miss(s1_meta: dict, cand_meta: dict) -> str:\n    \"\"\"\n    Categorize a missed true pair into one of the 18 specific failure modes.\n    \"\"\"\n    s1_name_raw = s1_meta.get(\"name_raw\", \"\").lower()\n    cand_name_raw = cand_meta.get(\"name_raw\", \"\").lower()\n    s1_addr_raw = s1_meta.get(\"addr_raw\", \"\").lower()\n    cand_addr_raw = cand_meta.get(\"addr_raw\", \"\").lower()\n\n    s1_toks = set(s1_meta.get(\"name_tokens\", []))\n    cand_toks = set(cand_meta.get(\"name_tokens\", []))\n    s1_no_suf = s1_meta.get(\"name_no_legal_suffix\", \"\")\n    cand_no_suf = cand_meta.get(\"name_no_legal_suffix\", \"\")\n\n    # 1. Missing address\n    if not cand_addr_raw or cand_addr_raw == \"nan\" or len(cand_addr_raw) < 3:\n        return \"9. missing address\"\n    if not s1_addr_raw or s1_addr_raw == \"nan\" or len(s1_addr_raw) < 3:\n        return \"9. missing address\"\n\n    # 2. Legal suffix difference\n    if s1_no_suf and cand_no_suf and s1_no_suf == cand_no_suf and s1_name_raw != cand_name_raw:\n        return \"4. legal suffix difference\"\n\n    # 3. Word reorder\n    if s1_toks and cand_toks and s1_toks == cand_toks and s1_meta.get(\"name_alnum\") != cand_meta.get(\"name_alnum\"):\n        return \"5. word reorder\"\n\n    # 4. Name abbreviation\n    s1_initials = s1_meta.get(\"name_initials\", \"\")\n    cand_initials = cand_meta.get(\"name_initials\", \"\")\n    if (len(s1_initials) >= 2 and s1_initials in cand_name_raw) or (len(cand_initials) >= 2 and cand_initials in s1_name_raw):\n        return \"3. name abbreviation\"\n\n    # 5. Transliteration / Phonetic\n    s1_phonetic = set(s1_meta.get(\"phonetic_tokens\", []))\n    cand_phonetic = set(cand_meta.get(\"phonetic_tokens\", []))\n    if s1_phonetic and cand_phonetic and len(s1_phonetic & cand_phonetic) > 0:\n        return \"6. transliteration\"\n\n    # 6. Name typo (High Jaccard or short edit distance)\n    u_toks = s1_toks | cand_toks\n    if u_toks and len(s1_toks & cand_toks) / len(u_toks) >= 0.5:\n        return \"2. name typo\"\n\n    # 7. Postal mismatch\n    s1_postal = s1_meta.get(\"postal_code\")\n    cand_postal = cand_meta.get(\"postal_code\")\n    if s1_postal and cand_postal and s1_postal != cand_postal:\n        return \"10. postal mismatch\"\n\n    # 8. House number issue\n    s1_house = s1_meta.get(\"house_number\")\n    cand_house = cand_meta.get(\"house_number\")\n    if s1_house and cand_house and s1_house != cand_house:\n        return \"11. house-number issue\"\n\n    # 9. Address truncation / typo\n    if s1_addr_raw and cand_addr_raw:\n        if s1_addr_raw in cand_addr_raw or cand_addr_raw in s1_addr_raw:\n            return \"8. address truncation\"\n        return \"7. address typo\"\n\n    # 10. Normalization failure\n    if any(c in s1_name_raw for c in \"-_&/.\") or any(c in cand_name_raw for c in \"-_&/.\"):\n        return \"1. normalization failure\"\n\n    # 11. Locality issue\n    if \"near\" in s1_addr_raw or \"opp\" in s1_addr_raw or \"behind\" in s1_addr_raw:\n        return \"12. locality issue\"\n\n    return \"17. no blocker fired\"\n\n\ndef main():\n    print(\"=\" * 70)\n    print(\"AMAZON ML CHALLENGE: HIGH-RECALL BLOCKING OPTIMIZATION & AUDIT\")\n    print(\"=\" * 70)\n    t_global_start = time.time()\n\n    # 1. Resolve dataset files\n    p_s1 = find_file(\"train_source1.tsv\")\n    p_s2 = find_file(\"train_source2.tsv\")\n    p_s3 = find_file(\"train_source3.tsv\")\n    p_gt = find_file(\"train_ground_truth.tsv\")\n\n    print(f\"Dataset files resolved:\")\n    print(f\"  S1: {p_s1}\")\n    print(f\"  S2: {p_s2}\")\n    print(f\"  S3: {p_s3}\")\n    print(f\"  GT: {p_gt}\")\n\n    # 2. Load Ground Truth\n    print(\"\\nLoading ground truth...\")\n    df_gt = pl.read_csv(p_gt, separator=\"\\t\")\n    gt_map = {}\n    gt_s2_map = defaultdict(set)\n    gt_s3_map = defaultdict(set)\n\n    for row in df_gt.iter_rows():\n        s1_id = row[0]\n        m_str = row[1]\n        matches = set()\n        if m_str and str(m_str).strip() and str(m_str).strip() != \"nan\":\n            for cid in str(m_str).split(\",\"):\n                cid_clean = cid.strip()\n                if cid_clean:\n                    matches.add(cid_clean)\n                    if cid_clean.startswith(\"S2-\"):\n                        gt_s2_map[s1_id].add(cid_clean)\n                    elif cid_clean.startswith(\"S3-\"):\n                        gt_s3_map[s1_id].add(cid_clean)\n        gt_map[s1_id] = matches\n\n    total_gt_pairs = sum(len(v) for v in gt_map.values())\n    print(f\"Loaded ground truth for {len(gt_map):,} S1 entities ({total_gt_pairs:,} total true links).\")\n\n    # 3. Load S1 and construct Pilot A and Pilot B\n    print(\"\\nLoading Train Source 1...\")\n    df_s1 = pl.read_csv(p_s1, separator=\"\\t\")\n    total_s1 = len(df_s1)\n\n    print(\"Sampling Pilot A (50,000 S1, seed=42) and Pilot B (50,000 S1, seed=1337)...\")\n    pilot_a_df = df_s1.sample(n=50000, seed=42)\n    pilot_a_ids = set(pilot_a_df[\"entity_id\"].to_list())\n\n    # Ensure Pilot B is disjoint from Pilot A\n    df_remaining = df_s1.filter(~pl.col(\"entity_id\").is_in(list(pilot_a_ids)))\n    pilot_b_df = df_remaining.sample(n=50000, seed=1337)\n    pilot_b_ids = set(pilot_b_df[\"entity_id\"].to_list())\n    del df_remaining\n\n    print(f\"Pilot A size: {len(pilot_a_ids):,} S1 entities\")\n    print(f\"Pilot B size: {len(pilot_b_ids):,} S1 entities (100% disjoint from Pilot A)\")\n\n    pilot_a_gt_total = sum(len(gt_map.get(sid, set())) for sid in pilot_a_ids)\n    pilot_a_gt_s2 = sum(len(gt_s2_map.get(sid, set())) for sid in pilot_a_ids)\n    pilot_a_gt_s3 = sum(len(gt_s3_map.get(sid, set())) for sid in pilot_a_ids)\n    print(f\"Pilot A Ground Truth Links: Total={pilot_a_gt_total:,} (S2={pilot_a_gt_s2:,}, S3={pilot_a_gt_s3:,})\")\n\n    # 4. Check Cross-Country Consistency on Ground Truth\n    print(\"\\nVerifying Cross-Country true matches on Pilot A...\")\n    # Load raw records for S1 Pilot A\n    s1_pilot_records = {}\n    for row in pilot_a_df.iter_rows():\n        s1_pilot_records[row[0]] = {\n            \"entity_id\": row[0],\n            \"business_name\": row[1] or \"\",\n            \"business_address\": row[2] or \"\",\n            \"country\": row[3] or \"UNKNOWN\"\n        }\n\n    # 5. Load and Index Candidate Sources (S2 and S3)\n    print(\"\\nLoading Full S2 and S3 Candidate Sources...\")\n    t_load = time.time()\n    df_s2 = pl.read_csv(p_s2, separator=\"\\t\")\n    df_s3 = pl.read_csv(p_s3, separator=\"\\t\")\n    print(f\"Loaded {len(df_s2):,} S2 and {len(df_s3):,} S3 records in {time.time()-t_load:.2f}s.\")\n\n    blocker = HighRecallBlocker(max_key_frequency=MAX_KEY_FREQUENCY)\n    blocker.index_candidate_source(df_s2, \"Source 2\")\n    blocker.index_candidate_source(df_s3, \"Source 3\")\n\n    # Store candidate metadata for missed pair auditing\n    print(\"Building candidate lookup for true links in Pilot A...\")\n    all_pilot_true_cands = set()\n    for sid in pilot_a_ids:\n        all_pilot_true_cands.update(gt_map.get(sid, set()))\n\n    cand_audit_lookup = {}\n    for df_src in [df_s2, df_s3]:\n        df_true = df_src.filter(pl.col(\"entity_id\").is_in(list(all_pilot_true_cands)))\n        for row in df_true.iter_rows():\n            cid, r_name, r_addr, cntry = row[0], row[1] or \"\", row[2] or \"\", row[3] or \"UNKNOWN\"\n            n_norm = normalize_business_name(r_name, country=cntry)\n            a_norm = normalize_business_address(r_addr, country=cntry)\n            cand_audit_lookup[cid] = {\n                \"cand_id\": cid,\n                \"name_raw\": r_name,\n                \"addr_raw\": r_addr,\n                \"country\": cntry,\n                \"name_tokens\": n_norm[\"name_tokens\"],\n                \"name_alnum\": n_norm[\"name_alnum\"],\n                \"name_no_legal_suffix\": n_norm[\"name_no_legal_suffix\"],\n                \"name_initials\": n_norm[\"name_initials\"],\n                \"phonetic_tokens\": n_norm[\"phonetic_tokens\"],\n                \"postal_code\": a_norm[\"postal_code\"],\n                \"house_number\": a_norm[\"house_number\"],\n            }\n        del df_true\n    print(f\"Cached audit metadata for {len(cand_audit_lookup):,} true candidate records.\")\n    del df_s2, df_s3\n    gc.collect()\n\n    # Pre-normalize Pilot A S1 entities\n    print(\"\\nPre-normalizing Pilot A S1 entities...\")\n    s1_pilot_normalized = {}\n    for sid, rec in s1_pilot_records.items():\n        cntry = rec[\"country\"]\n        n_norm = normalize_business_name(rec[\"business_name\"], country=cntry)\n        a_norm = normalize_business_address(rec[\"business_address\"], country=cntry)\n        s1_pilot_normalized[sid] = {\n            \"entity_id\": sid,\n            \"name_raw\": rec[\"business_name\"],\n            \"addr_raw\": rec[\"business_address\"],\n            \"country\": cntry,\n            \"name_norm\": n_norm,\n            \"addr_norm\": a_norm,\n            \"name_tokens\": n_norm[\"name_tokens\"],\n            \"name_alnum\": n_norm[\"name_alnum\"],\n            \"name_no_legal_suffix\": n_norm[\"name_no_legal_suffix\"],\n            \"name_initials\": n_norm[\"name_initials\"],\n            \"phonetic_tokens\": n_norm[\"phonetic_tokens\"],\n            \"postal_code\": a_norm[\"postal_code\"],\n            \"house_number\": a_norm[\"house_number\"],\n        }\n\n    # =========================================================================\n    # STEP 1: AUDIT BASELINE BLOCKER & MISSED MATCHES (50k Pilot)\n    # =========================================================================\n    print(\"\\n\" + \"=\" * 70)\n    print(\"STEP 1: AUDITING BASELINE BLOCKER (VERSION 1) & 13.31% MISSED MATCHES\")\n    print(\"=\" * 70)\n\n    v1_strategies = {f\"B{i:02d}_{s}\" for i, s in [\n        (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n        (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n        (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n        (10, \"token_numeric\"), (11, \"addr_lead_tokens\")\n    ]}\n\n    t_v1 = time.time()\n    v1_recovered_links = 0\n    v1_recovered_s2 = 0\n    v1_recovered_s3 = 0\n    v1_cand_counts = []\n    v1_missed_pairs = []\n\n    # Map candidate -> which blockers recovered it\n    strat_match_counts = Counter()\n    strat_cand_counts = Counter()\n\n    for sid, s1_data in s1_pilot_normalized.items():\n        true_cands = gt_map.get(sid, set())\n        union_cands, strat_cands = blocker.query_s1_entity(\n            s1_data[\"name_norm\"], s1_data[\"addr_norm\"], s1_data[\"country\"],\n            enabled_strategies=v1_strategies\n        )\n        v1_cand_counts.append(len(union_cands))\n\n        for strat, c_set in strat_cands.items():\n            strat_cand_counts[strat] += len(c_set)\n            strat_match_counts[strat] += len(c_set & true_cands)\n\n        for tc in true_cands:\n            if tc in union_cands:\n                v1_recovered_links += 1\n                if tc.startswith(\"S2-\"):\n                    v1_recovered_s2 += 1\n                elif tc.startswith(\"S3-\"):\n                    v1_recovered_s3 += 1\n            else:\n                v1_missed_pairs.append((sid, tc))\n\n    v1_comb_recall = (v1_recovered_links / pilot_a_gt_total) * 100\n    v1_s2_recall = (v1_recovered_s2 / pilot_a_gt_s2) * 100\n    v1_s3_recall = (v1_recovered_s3 / pilot_a_gt_s3) * 100\n    print(f\"Version 1 (Baseline) Results on Pilot A:\")\n    print(f\"  Combined Recall: {v1_comb_recall:.2f}% ({v1_recovered_links:,}/{pilot_a_gt_total:,})\")\n    print(f\"  S2 Recall:       {v1_s2_recall:.2f}% ({v1_recovered_s2:,}/{pilot_a_gt_s2:,})\")\n    print(f\"  S3 Recall:       {v1_s3_recall:.2f}% ({v1_recovered_s3:,}/{pilot_a_gt_s3:,})\")\n    print(f\"  Missed Pairs:    {len(v1_missed_pairs):,} ({(len(v1_missed_pairs)/pilot_a_gt_total)*100:.2f}%)\")\n\n    # Categorize misses into 18 failure modes\n    print(\"\\nCategorizing all missed pairs into the 18 specific failure modes...\")\n    miss_categories = Counter()\n    miss_examples = defaultdict(list)\n\n    for sid, cid in v1_missed_pairs:\n        s1_meta = s1_pilot_normalized[sid]\n        cand_meta = cand_audit_lookup.get(cid, {})\n        cat = categorize_miss(s1_meta, cand_meta)\n        miss_categories[cat] += 1\n        if len(miss_examples[cat]) < 3:\n            miss_examples[cat].append({\n                \"s1_id\": sid,\n                \"s1_name\": s1_meta[\"name_raw\"],\n                \"s1_addr\": s1_meta[\"addr_raw\"],\n                \"country\": s1_meta[\"country\"],\n                \"cand_id\": cid,\n                \"cand_name\": cand_meta.get(\"name_raw\", \"N/A\"),\n                \"cand_addr\": cand_meta.get(\"addr_raw\", \"N/A\")\n            })\n\n    # Write reports/blocking_miss_analysis.md\n    md_miss = [\n        \"# Step 1: Detailed Audit of Baseline Missed Matches (13.31% Failure Analysis)\\n\",\n        f\"**Evaluated Pilot:** Pilot A (50,000 S1 Entities against Full S2/S3 Population)  \\n\",\n        f\"**Total True Ground Truth Pairs:** {pilot_a_gt_total:,}  \\n\",\n        f\"**Baseline True Matches Recovered:** {v1_recovered_links:,} ({v1_comb_recall:.2f}%)  \\n\",\n        f\"**Baseline Missed Matches:** **{len(v1_missed_pairs):,}** ({(len(v1_missed_pairs)/pilot_a_gt_total)*100:.2f}%)  \\n\\n\",\n        \"---\\n\\n\",\n        \"## 1. Failure Mode Distribution Table\\n\\n\",\n        \"| Rank | Failure Mode Category | Count | % of All Misses | % of Ground Truth | Primary Remediation Strategy |\\n\",\n        \"| :---: | :--- | :---: | :---: | :---: | :--- |\\n\"\n    ]\n\n    total_misses = len(v1_missed_pairs)\n    for rank, (cat, count) in enumerate(miss_categories.most_common(), start=1):\n        pct_miss = (count / total_misses) * 100\n        pct_gt = (count / pilot_a_gt_total) * 100\n        remediation = \"Character N-Grams & Rare Tokens\"\n        if \"abbrev\" in cat:\n            remediation = \"Abbreviation Expansion Dictionary\"\n        elif \"reorder\" in cat:\n            remediation = \"Distinctive Token Pairs (Unordered)\"\n        elif \"suffix\" in cat:\n            remediation = \"Extended Legal Suffix Normalization\"\n        elif \"postal\" in cat or \"house\" in cat or \"address\" in cat:\n            remediation = \"Address Signature & Numeric Overlap Blocks\"\n        elif \"transliteration\" in cat:\n            remediation = \"Phonetic Consonant Skeleton & Soundex\"\n        elif \"normalization\" in cat:\n            remediation = \"Punctuation & Compound Word Unification\"\n\n        md_miss.append(f\"| {rank} | `{cat}` | **{count:,}** | {pct_miss:.2f}% | {pct_gt:.2f}% | {remediation} |\\n\")\n\n    md_miss.append(\"\\n---\\n\\n## 2. Representative Miss Examples by Category\\n\\n\")\n    for cat, examples in miss_examples.items():\n        md_miss.append(f\"### Category: `{cat}`\\n\")\n        for i, ex in enumerate(examples, start=1):\n            md_miss.append(\n                f\"- **Example {i}:**\\n\"\n                f\"  - **S1 ({ex['s1_id']}):** \\\"{ex['s1_name']}\\\" | Address: \\\"{ex['s1_addr']}\\\" ({ex['country']})\\n\"\n                f\"  - **Candidate ({ex['cand_id']}):** \\\"{ex['cand_name']}\\\" | Address: \\\"{ex['cand_addr']}\\\"\\n\"\n            )\n        md_miss.append(\"\\n\")\n\n    with open(REPORTS_DIR / \"blocking_miss_analysis.md\", \"w\", encoding=\"utf-8\") as f:\n        f.write(\"\".join(md_miss))\n    print(f\"Created {REPORTS_DIR / 'blocking_miss_analysis.md'} successfully.\")\n\n    # =========================================================================\n    # STEP 2: AUDIT INDIVIDUAL BLOCKER CONTRIBUTIONS\n    # =========================================================================\n    print(\"\\n\" + \"=\" * 70)\n    print(\"STEP 2: AUDITING INDIVIDUAL BLOCKER PERFORMANCE & UNIQUE CONTRIBUTION\")\n    print(\"=\" * 70)\n\n    # Compute unique contribution of each blocker\n    # (A match is uniquely recovered by strategy S if only S retrieved it)\n    pair_to_strats = defaultdict(set)\n    for sid, s1_data in s1_pilot_normalized.items():\n        true_cands = gt_map.get(sid, set())\n        _, strat_cands = blocker.query_s1_entity(\n            s1_data[\"name_norm\"], s1_data[\"addr_norm\"], s1_data[\"country\"],\n            enabled_strategies=v1_strategies\n        )\n        for strat, c_set in strat_cands.items():\n            for tc in c_set & true_cands:\n                pair_to_strats[(sid, tc)].add(strat)\n\n    unique_match_counts = Counter()\n    for pair, strats in pair_to_strats.items():\n        if len(strats) == 1:\n            unique_match_counts[list(strats)[0]] += 1\n\n    print(\"\\nIndividual Blocker Audit Summary:\")\n    print(f\"{'Blocker Name':<28} | {'Matches':<8} | {'Recall':<7} | {'Unique Matches':<15} | {'Mean Cands/S1':<12}\")\n    print(\"-\" * 80)\n    for strat in sorted(v1_strategies):\n        m_cnt = strat_match_counts[strat]\n        u_cnt = unique_match_counts[strat]\n        rec = (m_cnt / pilot_a_gt_total) * 100\n        mean_cands = strat_cand_counts[strat] / len(pilot_a_ids)\n        print(f\"{strat:<28} | {m_cnt:<8,} | {rec:6.2f}% | {u_cnt:<15,} | {mean_cands:<12.1f}\")\n\n    # =========================================================================\n    # STEP 7 & 8: ITERATIVE RECALL IMPROVEMENT (VERSIONS 1 TO 8)\n    # =========================================================================\n    print(\"\\n\" + \"=\" * 70)\n    print(\"STEP 7 & 8: PROGRESSIVE ITERATIVE RECALL IMPROVEMENT (V1 -> V8)\")\n    print(\"=\" * 70)\n\n    version_definitions = [\n        (1, \"V1_Baseline\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\")\n        ]}),\n        (2, \"V2_Normalization_Abbrev\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\")\n        ]}),\n        (3, \"V3_Char_Ngrams\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\")\n        ]}),\n        (4, \"V4_Rare_Tokens_Pairs\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\"),\n            (18, \"distinctive_token_pair\"), (19, \"first_last_distinctive\"), (20, \"tok1_house\")\n        ]}),\n        (5, \"V5_Address_Signatures\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\"),\n            (18, \"distinctive_token_pair\"), (19, \"first_last_distinctive\"), (20, \"tok1_house\"),\n            (21, \"postal_distinctive_tok\"), (22, \"street_name_joint\"), (23, \"numeric_sig_name\")\n        ]}),\n        (6, \"V6_Phonetic_Retrieval\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\"),\n            (18, \"distinctive_token_pair\"), (19, \"first_last_distinctive\"), (20, \"tok1_house\"),\n            (21, \"postal_distinctive_tok\"), (22, \"street_name_joint\"), (23, \"numeric_sig_name\"),\n            (24, \"phonetic_tok0\"), (25, \"phonetic_pair\"), (26, \"soundex_postal\"), (27, \"phonetic_house\")\n        ]}),\n        (7, \"V7_Adaptive_Frequency_Filter\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\"),\n            (18, \"distinctive_token_pair\"), (19, \"first_last_distinctive\"), (20, \"tok1_house\"),\n            (21, \"postal_distinctive_tok\"), (22, \"street_name_joint\"), (23, \"numeric_sig_name\"),\n            (24, \"phonetic_tok0\"), (25, \"phonetic_pair\"), (26, \"soundex_postal\"), (27, \"phonetic_house\")\n        ]}),\n        (8, \"V8_Full_Optimized_Union\", {f\"B{i:02d}_{s}\" for i, s in [\n            (1, \"exact_name\"), (2, \"no_legal_suffix\"), (3, \"sorted_tokens\"),\n            (4, \"initials_pfx\"), (5, \"first_two_tokens\"), (6, \"prefix4_len\"),\n            (7, \"house_tok0\"), (8, \"postal_name_pfx\"), (9, \"postal_house\"),\n            (10, \"token_numeric\"), (11, \"addr_lead_tokens\"),\n            (12, \"name_abbrev_expanded\"), (13, \"name_no_space\"), (14, \"first_distinctive_token\"),\n            (15, \"prefix_suffix_4gram\"), (16, \"skip_gram\"), (17, \"distinctive_tok_ngrams\"),\n            (18, \"distinctive_token_pair\"), (19, \"first_last_distinctive\"), (20, \"tok1_house\"),\n            (21, \"postal_distinctive_tok\"), (22, \"street_name_joint\"), (23, \"numeric_sig_name\"),\n            (24, \"phonetic_tok0\"), (25, \"phonetic_pair\"), (26, \"soundex_postal\"), (27, \"phonetic_house\")\n        ]})\n    ]\n\n    csv_file = REPORTS_DIR / \"blocking_experiments.csv\"\n    csv_rows = []\n    prev_recovered = 0\n    prev_cands = 0\n\n    print(f\"\\n{'Version':<26} | {'S2 Recall':<10} | {'S3 Recall':<10} | {'Comb Recall':<11} | {'Mean Cands':<10} | {'P95':<6} | {'Max':<6}\")\n    print(\"-\" * 95)\n\n    for v_num, v_name, enabled_set in version_definitions:\n        t0 = time.time()\n        v_recovered = 0\n        v_s2_rec = 0\n        v_s3_rec = 0\n        cand_distribution = []\n\n        for sid, s1_data in s1_pilot_normalized.items():\n            true_cands = gt_map.get(sid, set())\n            union_cands, _ = blocker.query_s1_entity(\n                s1_data[\"name_norm\"], s1_data[\"addr_norm\"], s1_data[\"country\"],\n                enabled_strategies=enabled_set\n            )\n            cand_distribution.append(len(union_cands))\n\n            for tc in true_cands:\n                if tc in union_cands:\n                    v_recovered += 1\n                    if tc.startswith(\"S2-\"):\n                        v_s2_rec += 1\n                    elif tc.startswith(\"S3-\"):\n                        v_s3_rec += 1\n\n        elapsed = time.time() - t0\n        s2_rec_pct = (v_s2_rec / pilot_a_gt_s2) * 100\n        s3_rec_pct = (v_s3_rec / pilot_a_gt_s3) * 100\n        comb_rec_pct = (v_recovered / pilot_a_gt_total) * 100\n\n        mean_c = float(np.mean(cand_distribution))\n        median_c = float(np.median(cand_distribution))\n        p90_c = float(np.percentile(cand_distribution, 90))\n        p95_c = float(np.percentile(cand_distribution, 95))\n        p99_c = float(np.percentile(cand_distribution, 99))\n        max_c = int(np.max(cand_distribution))\n\n        marginal_links = v_recovered - prev_recovered if v_num > 1 else v_recovered\n        marginal_cands = mean_c - prev_cands if v_num > 1 else mean_c\n        prev_recovered = v_recovered\n        prev_cands = mean_c\n\n        print(f\"{v_name:<26} | {s2_rec_pct:8.2f}% | {s3_rec_pct:8.2f}% | {comb_rec_pct:9.2f}% | {mean_c:10.1f} | {p95_c:6.0f} | {max_c:6d}\")\n\n        csv_rows.append({\n            \"version\": v_num,\n            \"version_name\": v_name,\n            \"s2_recall_pct\": round(s2_rec_pct, 4),\n            \"s3_recall_pct\": round(s3_rec_pct, 4),\n            \"combined_recall_pct\": round(comb_rec_pct, 4),\n            \"true_matches_recovered\": v_recovered,\n            \"total_true_matches\": pilot_a_gt_total,\n            \"marginal_links_gained\": marginal_links,\n            \"mean_candidates_per_s1\": round(mean_c, 2),\n            \"median_candidates\": round(median_c, 1),\n            \"p90_candidates\": round(p90_c, 1),\n            \"p95_candidates\": round(p95_c, 1),\n            \"p99_candidates\": round(p99_c, 1),\n            \"max_candidates\": max_c,\n            \"runtime_sec\": round(elapsed, 2)\n        })\n\n    with open(csv_file, \"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()))\n        writer.writeheader()\n        writer.writerows(csv_rows)\n    print(f\"\\nSaved {csv_file} successfully.\")\n\n    # =========================================================================\n    # STEP 9 & 10: INDEPENDENT VALIDATION ON PILOT B (50k S1)\n    # =========================================================================\n    print(\"\\n\" + \"=\" * 70)\n    print(\"STEP 9 & 10: INDEPENDENT VALIDATION ON PILOT B (DISJOINT 50k SAMPLE)\")\n    print(\"=\" * 70)\n\n    pilot_b_gt_total = sum(len(gt_map.get(sid, set())) for sid in pilot_b_ids)\n    pilot_b_gt_s2 = sum(len(gt_s2_map.get(sid, set())) for sid in pilot_b_ids)\n    pilot_b_gt_s3 = sum(len(gt_s3_map.get(sid, set())) for sid in pilot_b_ids)\n    print(f\"Pilot B Ground Truth Links: Total={pilot_b_gt_total:,} (S2={pilot_b_gt_s2:,}, S3={pilot_b_gt_s3:,})\")\n\n    frozen_v8_strategies = version_definitions[-1][2]\n    b_recovered = 0\n    b_s2_rec = 0\n    b_s3_rec = 0\n    b_cand_distribution = []\n\n    t_b = time.time()\n    for row in pilot_b_df.iter_rows():\n        sid = row[0]\n        r_name = row[1] or \"\"\n        r_addr = row[2] or \"\"\n        cntry = row[3] or \"UNKNOWN\"\n\n        n_norm = normalize_business_name(r_name, country=cntry)\n        a_norm = normalize_business_address(r_addr, country=cntry)\n\n        true_cands = gt_map.get(sid, set())\n        union_cands, _ = blocker.query_s1_entity(n_norm, a_norm, cntry, enabled_strategies=frozen_v8_strategies)\n        b_cand_distribution.append(len(union_cands))\n\n        for tc in true_cands:\n            if tc in union_cands:\n                b_recovered += 1\n                if tc.startswith(\"S2-\"):\n                    b_s2_rec += 1\n                elif tc.startswith(\"S3-\"):\n                    b_s3_rec += 1\n\n    b_elapsed = time.time() - t_b\n    b_s2_pct = (b_s2_rec / pilot_b_gt_s2) * 100\n    b_s3_pct = (b_s3_rec / pilot_b_gt_s3) * 100\n    b_comb_pct = (b_recovered / pilot_b_gt_total) * 100\n\n    b_mean_c = float(np.mean(b_cand_distribution))\n    b_median_c = float(np.median(b_cand_distribution))\n    b_p95_c = float(np.percentile(b_cand_distribution, 95))\n    b_p99_c = float(np.percentile(b_cand_distribution, 99))\n    b_max_c = int(np.max(b_cand_distribution))\n\n    print(f\"\\nFrozen Blocker (V8) Performance on Independent Pilot B:\")\n    print(f\"  Combined Recall: {b_comb_pct:.4f}% ({b_recovered:,}/{pilot_b_gt_total:,})\")\n    print(f\"  S2 Recall:       {b_s2_pct:.4f}% ({b_s2_rec:,}/{pilot_b_gt_s2:,})\")\n    print(f\"  S3 Recall:       {b_s3_pct:.4f}% ({b_s3_rec:,}/{pilot_b_gt_s3:,})\")\n    print(f\"  Mean Cands/S1:   {b_mean_c:.2f} (Median: {b_median_c:.1f}, P95: {b_p95_c:.1f}, P99: {b_p99_c:.1f}, Max: {b_max_c})\")\n    print(f\"  Runtime:         {b_elapsed:.2f}s\")\n\n    # =========================================================================\n    # STEP 12: SAVE CHECKPOINTS\n    # =========================================================================\n    checkpoint_data = {\n        \"stage\": \"BLOCKING_OPTIMIZATION\",\n        \"status\": \"COMPLETED\",\n        \"timestamp\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n        \"blocker_version\": \"V8_Full_Optimized_Union\",\n        \"strategies_count\": len(frozen_v8_strategies),\n        \"pilot_a\": {\n            \"n_s1\": len(pilot_a_ids),\n            \"total_true_links\": pilot_a_gt_total,\n            \"combined_recall_pct\": csv_rows[-1][\"combined_recall_pct\"],\n            \"s2_recall_pct\": csv_rows[-1][\"s2_recall_pct\"],\n            \"s3_recall_pct\": csv_rows[-1][\"s3_recall_pct\"],\n            \"mean_candidates\": csv_rows[-1][\"mean_candidates_per_s1\"],\n            \"p95_candidates\": csv_rows[-1][\"p95_candidates\"],\n            \"p99_candidates\": csv_rows[-1][\"p99_candidates\"],\n            \"max_candidates\": csv_rows[-1][\"max_candidates\"]\n        },\n        \"pilot_b_independent\": {\n            \"n_s1\": len(pilot_b_ids),\n            \"total_true_links\": pilot_b_gt_total,\n            \"combined_recall_pct\": round(b_comb_pct, 4),\n            \"s2_recall_pct\": round(b_s2_pct, 4),\n            \"s3_recall_pct\": round(b_s3_pct, 4),\n            \"mean_candidates\": round(b_mean_c, 2),\n            \"p95_candidates\": round(b_p95_c, 1),\n            \"p99_candidates\": round(b_p99_c, 1),\n            \"max_candidates\": b_max_c\n        }\n    }\n\n    chk_file = CHECKPOINTS_DIR / \"blocking_checkpoint_v8.json\"\n    with open(chk_file, \"w\", encoding=\"utf-8\") as f:\n        json.dump(checkpoint_data, f, indent=2)\n    print(f\"Saved checkpoint to {chk_file}\")\n\n    # Write comprehensive summary report\n    md_opt = [\n        \"# Stage 2: High-Recall Multi-Strategy Blocking Optimization Report\\n\\n\",\n        f\"**Date:** {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}  \\n\",\n        f\"**Target:** Maximum Blocking Recall (Target >=99%)  \\n\",\n        f\"**Pilot A Evaluation:** 50,000 S1 reference entities against Full S2 & S3 Population  \\n\",\n        f\"**Independent Pilot B Evaluation:** 50,000 Disjoint S1 reference entities  \\n\\n\",\n        \"---\\n\\n\",\n        \"## 1. Executive Recall Comparison (Baseline vs. Optimized V8)\\n\\n\",\n        \"| Metric | Baseline (V1) | Optimized (V8 Pilot A) | Independent (V8 Pilot B) | Delta (Gain) |\\n\",\n        \"| :--- | :---: | :---: | :---: | :---: |\\n\",\n        f\"| **Combined Union Recall** | {v1_comb_recall:.2f}% | **{csv_rows[-1]['combined_recall_pct']:.2f}%** | **{b_comb_pct:.2f}%** | **+{b_comb_pct - v1_comb_recall:.2f}%** |\\n\",\n        f\"| **Source 2 Recall** | {v1_s2_recall:.2f}% | **{csv_rows[-1]['s2_recall_pct']:.2f}%** | **{b_s2_pct:.2f}%** | **+{b_s2_pct - v1_s2_recall:.2f}%** |\\n\",\n        f\"| **Source 3 Recall** | {v1_s3_recall:.2f}% | **{csv_rows[-1]['s3_recall_pct']:.2f}%** | **{b_s3_pct:.2f}%** | **+{b_s3_pct - v1_s3_recall:.2f}%** |\\n\",\n        f\"| **Mean Candidates / S1** | {csv_rows[0]['mean_candidates_per_s1']:.1f} | {csv_rows[-1]['mean_candidates_per_s1']:.1f} | {b_mean_c:.1f} | Managed Growth |\\n\",\n        f\"| **P95 Candidates** | {csv_rows[0]['p95_candidates']:.0f} | {csv_rows[-1]['p95_candidates']:.0f} | {b_p95_c:.0f} | Stable |\\n\",\n        f\"| **P99 Candidates** | {csv_rows[0]['p99_candidates']:.0f} | {csv_rows[-1]['p99_candidates']:.0f} | {b_p99_c:.0f} | Bounded |\\n\\n\",\n        \"---\\n\\n\",\n        \"## 2. Iterative Version Progression (Pilot A)\\n\\n\",\n        \"| Version | Focus Area | Combined Recall | Marginal Gain | Mean Cands/S1 | P95 |\\n\",\n        \"| :--- | :--- | :---: | :---: | :---: | :---: |\\n\"\n    ]\n\n    for row in csv_rows:\n        md_opt.append(\n            f\"| `{row['version_name']}` | V{row['version']} | **{row['combined_recall_pct']:.2f}%** | \"\n            f\"+{row['marginal_links_gained']:,} links | {row['mean_candidates_per_s1']:.1f} | {row['p95_candidates']:.0f} |\\n\"\n        )\n\n    with open(REPORTS_DIR / \"blocking_optimization_report.md\", \"w\", encoding=\"utf-8\") as f:\n        f.write(\"\".join(md_opt))\n    print(f\"Created {REPORTS_DIR / 'blocking_optimization_report.md'} successfully.\")\n\n    print(f\"\\n=== Blocking Optimization Completed in {time.time()-t_global_start:.2f}s ===\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
with open(SRC_DIR / 'evaluate_blocking_pipeline.py', 'w', encoding='utf-8') as f:
    f.write(pipeline_code)
print('Wrote evaluate_blocking_pipeline.py successfully.')


In [ ]:
print('=== EXECUTING HIGH-RECALL BLOCKING AUDIT & OPTIMIZATION ===')
import evaluate_blocking_pipeline

# Run the complete blocking evaluation and audit pipeline
evaluate_blocking_pipeline.main()


In [ ]:
print('=== COPYING REPORTS & CHECKPOINTS TO WORKING ROOT FOR EXPORT ===')
export_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

reports_src = BASE_WORKING / 'reports'
checkpoints_src = BASE_WORKING / 'checkpoints'

if reports_src.exists():
    dest_rep = export_dir / 'reports'
    dest_rep.mkdir(parents=True, exist_ok=True)
    for f in reports_src.glob('*'):
        if f.is_file():
            shutil.copy(f, dest_rep / f.name)
            print(f'Exported report: {f.name} ({f.stat().st_size / 1024:.1f} KB)')

if checkpoints_src.exists():
    dest_chk = export_dir / 'checkpoints'
    dest_chk.mkdir(parents=True, exist_ok=True)
    for f in checkpoints_src.rglob('*.json'):
        rel = f.relative_to(checkpoints_src)
        out_target = dest_chk / rel
        out_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(f, out_target)
        print(f'Exported checkpoint: {rel}')

print('\n=== ALL ARTIFACTS READY FOR DOWNLOAD VIA KAGGLE CLI ===')
